# BME 574 — Project 3 starter
### Five steps. Five cells for you to write. Everything else is done.

You are given a biomedical classification dataset, already assembled into a matrix with a split
that holds out **whole subjects**. Your job is to train a dense neural network on it, compare it
honestly against two baselines, and report what you find.

**Your job is the five cells marked `TODO`** — about twenty-five lines of PyTorch in total.

---

### What you hand in

1. **This notebook**, run top to bottom, all five `TODO` cells filled, every check passing.
2. **`REPORT.md`** — one page, the six questions in Step 6.
3. **`AI_USE.md`** — which AI tools you used and for what.

### Before you start

Do **`BME574_Project3_explained.ipynb`** (~30 min, read and run). It covers the three things this
project is graded on that the lecture did not: subject-level splitting, per-class metrics, and how
many seeds you need before you are allowed to claim a difference.

If you have not used PyTorch, do `BME574_pytorch_toolchain.ipynb` first.

### The one idea to hold on to

> **Your result is not a number, it is a comparison.** A network that scores 0.95 has told you
> nothing until you know what a straight-boundary model scores on the same split, what always
> guessing the majority class scores, and how much your own answer moves when you change the
> random seed. One of the three tracks in this project does **not** beat its baseline. Finding
> that out and saying so clearly is worth full marks.

---
## Step 0 · Load your bundle

Your track arrives as a single `.npz`. Until it does, everything below runs on `demo.npz`, a small
synthetic stand-in with the same structure — use it to get your code working, but never report
results from it.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

BUNDLE = 'bundles/demo.npz'     # <-- change to your track: har / thyroid / epilepsy
SEED = 0

torch.manual_seed(SEED)
plt.rcParams.update({'font.size': 10, 'figure.dpi': 110})
np.set_printoptions(precision=4, suppress=True)

# progress flags - each TODO sets its own to True once its checks pass
ok1 = ok2 = ok3 = ok4 = ok5 = False

print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

torch 2.13.0 | CUDA False


In [2]:
def load_bundle(path):
    z = np.load(path, allow_pickle=True)
    b = {k: z[k] for k in z.files}
    for k in ('track', 'split_kind', 'source', 'notes'):
        b[k] = str(b[k])
    b['feature_names'] = [str(s) for s in b['feature_names']]
    b['class_names'] = [str(s) for s in b['class_names']]
    return b


if not os.path.exists(BUNDLE):
    raise SystemExit('%s not found - ask for your track bundle.' % BUNDLE)

B = load_bundle(BUNDLE)
X, y, subject = B['X'], B['y'], B['subject']
train_idx, test_idx = B['train_idx'], B['test_idx']
CLASSES = B['class_names']
K = len(CLASSES)

print('track       :', B['track'])
print('X           :', X.shape, X.dtype, ' ->', X.shape[1], 'features')
print('classes     :', K, '->', CLASSES)
print('subjects    :', len(np.unique(subject)))
print('split       :', B['split_kind'])
print('              %d train / %d test rows' % (len(train_idx), len(test_idx)))
print()
print(B['notes'])

track       : demo
X           : (985, 24) float32  -> 24 features
classes     : 3 -> ['class A', 'class B', 'class C']
subjects    : 60
split       : subject-wise split
              734 train / 251 test rows

Synthetic stand-in with the same shape as a real track: a curved class boundary the straight-boundary model cannot follow, a per-subject nuisance offset (so a random row split leaks), and one rare class. Use it to check that your code runs before your real bundle arrives. Do not report results from it.


In [3]:
counts = np.bincount(y, minlength=K)
print('%-16s %8s %8s %10s %9s' % ('class', 'n', 'share', 'n train', 'n test'))
for c in range(K):
    print('%-16s %8d %7.1f%% %10d %9d'
          % (CLASSES[c], counts[c], 100 * counts[c] / len(y),
             (y[train_idx] == c).sum(), (y[test_idx] == c).sum()))

majority_class = int(np.bincount(y[train_idx]).argmax())
majority_acc = float((y[test_idx] == majority_class).mean())
print()
print('BASELINE 1 - always answer "%s": test accuracy %.4f'
      % (CLASSES[majority_class], majority_acc))
print()
shared = np.intersect1d(subject[train_idx], subject[test_idx])
print('subjects appearing in BOTH splits: %d  <- must be 0' % len(shared))

class                   n    share    n train    n test
class A               417    42.3%        303       114
class B               396    40.2%        292       104
class C               172    17.5%        139        33

BASELINE 1 - always answer "class A": test accuracy 0.4542

subjects appearing in BOTH splits: 0  <- must be 0


### Standardise — on the training rows only

The scaler is a model parameter. Fitting it on all the data leaks the test set into your model and
every number after that is fiction.

In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X[train_idx])
Xtr = torch.tensor(scaler.transform(X[train_idx]), dtype=torch.float32)
Xte = torch.tensor(scaler.transform(X[test_idx]), dtype=torch.float32)
ytr = torch.tensor(y[train_idx], dtype=torch.long)
yte = torch.tensor(y[test_idx], dtype=torch.long)

print('Xtr', tuple(Xtr.shape), Xtr.dtype, '  ytr', tuple(ytr.shape), ytr.dtype)
print('training mean of feature 0: %+.6f  <- 0 by construction' % Xtr[:, 0].mean())
print('TEST     mean of feature 0: %+.6f  <- not 0, and that is correct' % Xte[:, 0].mean())


def check(name, cond, hint=''):
    print(f'  {"OK  " if cond else "NOT YET"}  {name}')
    if not cond and hint:
        print(f'           hint: {hint}')
    return bool(cond)

Xtr (734, 24) torch.float32   ytr (734,) torch.int64
training mean of feature 0: -0.000000  <- 0 by construction
TEST     mean of feature 0: -0.020831  <- not 0, and that is correct


### Baseline 2 — the best straight-boundary model

Multinomial logistic regression finds the best decision boundary that is a **straight cut** in
feature space. It is the thing your network has to beat to be worth its extra complexity. One line,
written for you.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix

linear = LogisticRegression(max_iter=3000).fit(Xtr.numpy(), ytr.numpy())
pred_linear = linear.predict(Xte.numpy())

print('BASELINE 2 - best straight-boundary model')
print('   accuracy %.4f   macro-F1 %.4f'
      % ((pred_linear == yte.numpy()).mean(),
         f1_score(yte.numpy(), pred_linear, average='macro')))
print('   per-class recall:')
for c in range(K):
    print('      %-16s %.3f'
          % (CLASSES[c], recall_score(yte.numpy(), pred_linear, labels=[c], average='macro')))

BASELINE 2 - best straight-boundary model
   accuracy 0.6693   macro-F1 0.6483
   per-class recall:
      class A          0.807
      class B          0.529
      class C          0.636


---
## Step 1 · **TODO 1** — build the network

Write a function that returns an `nn.Module` with this architecture:

```
input (p features)  ->  Linear -> ReLU -> Dropout
                    ->  Linear -> ReLU -> Dropout
                    ->  Linear -> K logits          (no softmax here!)
```

with hidden widths `(64, 32)` by default. `nn.Sequential` is enough — you do not need a subclass.

Two things the checks below will catch:

- **No softmax at the end.** `nn.CrossEntropyLoss` applies `log_softmax` internally. Add your own
  and you will softmax twice, which trains a measurably worse model without raising anything.
- **Dropout goes after the ReLU**, not after the final layer.

In [ ]:
def build_model(n_features, n_classes, hidden=(64, 32), p_drop=0.3, seed=SEED):
    torch.manual_seed(seed)
    # ---- TODO 1 ----------------------------------------------------------
    model = None
    # ----------------------------------------------------------------------
    return model


model = build_model(Xtr.shape[1], K)

n_par = sum(p.numel() for p in model.parameters()) if model is not None else 0
expected = (Xtr.shape[1] * 64 + 64) + (64 * 32 + 32) + (32 * K + K)

ok1 = all([
    check('build_model returns an nn.Module',
          isinstance(model, nn.Module),
          'nn.Sequential(nn.Linear(...), nn.ReLU(), nn.Dropout(...), ...) is a Module'),
    check('it maps p features to K logits',
          isinstance(model, nn.Module)
          and tuple(model(Xtr[:4]).shape) == (4, K),
          'the last Linear must have out_features = K = %d' % K),
    check('parameter count matches the architecture',
          n_par == expected,
          'expected %d = (%d*64+64) + (64*32+32) + (32*%d+%d)'
          % (expected, Xtr.shape[1], K, K)),
    check('the output is logits, not probabilities',
          isinstance(model, nn.Module)
          and not torch.allclose(model(Xtr[:8]).exp().sum(1), torch.ones(8), atol=1e-3),
          'remove the softmax - CrossEntropyLoss applies it for you'),
    check('dropout is present and active in train() mode',
          isinstance(model, nn.Module) and any(isinstance(m, nn.Dropout) for m in model.modules())
          and (model.train() or True)
          and not torch.equal(model(Xtr[:8]), model(Xtr[:8])),
          'nn.Dropout(p_drop) after each ReLU'),
])
print()
print('%d parameters.' % n_par if ok1 else 'Fix TODO 1 before continuing.')

---
## Step 2 · **TODO 2** — the training loop

Five lines inside the batch loop, in this order:

```python
optimiser.zero_grad()              # clear the gradients from the previous step
logits = model(xb)                 # forward; the graph is built here
loss = lossfn(logits, yb)          # one number, still attached to the graph
loss.backward()                    # walk the graph, fill every .grad
optimiser.step()                   # read .grad, update every parameter
```

`model.train()` and `model.eval()` are already called for you — they switch dropout on and off, and
getting that wrong is the single most common way to report a score that is quietly wrong.

In [ ]:
def evaluate(model, Xe, ye, lossfn):
    model.eval()                                  # dropout OFF
    with torch.no_grad():                         # no graph, no gradient bookkeeping
        logits = model(Xe)
        return lossfn(logits, ye).item(), logits.argmax(1)


def train_model(model, Xa, ya, Xv, yv, epochs=40, lr=1e-3, batch=128,
                weight=None, seed=SEED, verbose=True):
    loader = DataLoader(TensorDataset(Xa, ya), batch_size=batch, shuffle=True,
                        generator=torch.Generator().manual_seed(seed + 100))
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    lossfn = nn.CrossEntropyLoss(weight=weight)
    history = []

    for ep in range(epochs):
        model.train()                             # dropout ON
        for xb, yb in loader:
            # ---- TODO 2 --------------------------------------------------
            pass
            # ---------------------------------------------------------------

        tr_loss, tr_pred = evaluate(model, Xa, ya, lossfn)
        va_loss, va_pred = evaluate(model, Xv, yv, lossfn)
        history.append((tr_loss, va_loss,
                        (tr_pred == ya).double().mean().item(),
                        (va_pred == yv).double().mean().item()))
        if verbose and (ep < 2 or (ep + 1) % 10 == 0):
            print('epoch %3d   train loss %.4f  acc %.3f    |    test loss %.4f  acc %.3f'
                  % (ep + 1, history[-1][0], history[-1][2],
                     history[-1][1], history[-1][3]))
    return np.array(history)


if not ok1:
    raise SystemExit('Finish TODO 1 first.')

model = build_model(Xtr.shape[1], K)
before = evaluate(model, Xte, yte, nn.CrossEntropyLoss())[0]
history = train_model(model, Xtr, ytr, Xte, yte, epochs=40)
after = history[-1, 1]

ok2 = all([
    check('the loop ran for every epoch', history.shape == (40, 4)),
    check('the training loss went down',
          history[-1, 0] < history[0, 0],
          'if nothing changed, the five lines are still missing'),
    check('the model actually learned something',
          history[-1, 3] > majority_acc - 1e-9,
          'test accuracy should at least reach the majority baseline (%.3f)' % majority_acc),
    check('the loss is lower than before training',
          after < before,
          'untrained loss was %.4f' % before),
])
print()
print('Step 2 complete.' if ok2 else 'Fix TODO 2.')

### Figure 1 — the training curves *(written for you)*

Read the **gap** between the two curves, not the training curve. And read the loss panel before the
accuracy panel: the loss usually turns bad first.

In [ ]:
if not ok2:
    raise SystemExit('Finish TODO 2 first - Figure 1 needs the history.')

ep = np.arange(1, len(history) + 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(ep, history[:, 0], color='#1F3864', label='train')
ax[0].plot(ep, history[:, 1], color='#9C2B2B', label='test')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('cross-entropy'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, history[:, 2], color='#1F3864', label='train')
ax[1].plot(ep, history[:, 3], color='#9C2B2B', label='test')
ax[1].axhline(majority_acc, color='#2E6B4F', ls='--', lw=1.2)
ax[1].text(1.5, majority_acc - 0.03, 'majority class', color='#2E6B4F', fontsize=8)
ax[1].set_xlabel('epoch'); ax[1].set_ylabel('accuracy'); ax[1].legend(loc='lower right')
ax[1].grid(alpha=.3)
fig.suptitle('%s - %d parameters' % (B['track'], n_par), y=1.02)
plt.show()

---
## Step 3 · **TODO 3** — score it properly

Overall accuracy is one number and it is the least informative one you have. Fill in a function
that returns four things, so the rest of the notebook can use them.

- **accuracy** — the fraction correct
- **macro-F1** — the mean of the per-class F1 scores, so every class counts equally regardless of
  how rare it is
- **per-class recall** — of all the recordings that truly are class *c*, what fraction did we
  find? This is the number a clinician cares about
- **the confusion matrix** — rows are truth, columns are prediction

`sklearn.metrics` is imported for you; you are not expected to implement these from scratch.

In [ ]:
def score(y_true, y_pred, n_classes=K):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    # ---- TODO 3 ----------------------------------------------------------
    acc = None          # overall accuracy, a float
    macro_f1 = None     # f1_score(..., average='macro')
    recalls = None      # list of length n_classes, recall for each class in order
    cm = None           # confusion_matrix(y_true, y_pred)
    # ----------------------------------------------------------------------
    return acc, macro_f1, recalls, cm


if not ok2:
    raise SystemExit('Finish TODO 2 first.')

_, pred_net = evaluate(model, Xte, yte, nn.CrossEntropyLoss())
pred_net = pred_net.numpy()
acc, mf1, rec, cm = score(yte.numpy(), pred_net)

ref_rec = [recall_score(yte.numpy(), pred_net, labels=[c], average='macro') for c in range(K)]
ok3 = all([
    check('accuracy is right',
          acc is not None and abs(acc - (pred_net == yte.numpy()).mean()) < 1e-9),
    check('macro-F1 is right',
          mf1 is not None
          and abs(mf1 - f1_score(yte.numpy(), pred_net, average='macro')) < 1e-9,
          "f1_score(y_true, y_pred, average='macro')"),
    check('one recall per class, in class order',
          recalls_ok := (rec is not None and len(rec) == K
                         and np.allclose(np.asarray(rec, float), ref_rec)),
          'recall_score(y_true, y_pred, labels=[c], average="macro") for each c'),
    check('confusion matrix is K x K',
          cm is not None and np.asarray(cm).shape == (K, K)),
])

if ok3:
    print()
    print('%-16s %9s' % ('class', 'recall'))
    for c in range(K):
        print('%-16s %9.3f' % (CLASSES[c], rec[c]))
    print()
    print('accuracy %.4f   macro-F1 %.4f' % (acc, mf1))

### Figure 2 — the confusion matrix *(written for you)*

In [ ]:
if not ok3:
    raise SystemExit('Finish TODO 3 first - Figure 2 needs the confusion matrix.')

cmn = np.asarray(cm, float)
cmn = cmn / cmn.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(1.3 + 0.9 * K, 1.1 + 0.8 * K))
im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(K)); ax.set_xticklabels(CLASSES, rotation=30, ha='right')
ax.set_yticks(range(K)); ax.set_yticklabels(CLASSES)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(K):
    for j in range(K):
        ax.text(j, i, '%d\n%.2f' % (np.asarray(cm)[i, j], cmn[i, j]), ha='center',
                va='center', fontsize=8.5,
                color='white' if cmn[i, j] > 0.55 else '#14264A')
ax.set_title('row-normalised: each row sums to 1')
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

---
## Step 4 · **TODO 4** — the comparison, across seeds

This is the result. Everything before it was setup.

You have three candidates: the majority-class rule, the straight-boundary model, and your network.
But a single training run is not evidence — the network's initialisation and batch order are
random, and on some tracks that moves the accuracy by more than the difference you are trying to
measure.

**Train the network from three different seeds** and collect the scores, so you can report a
spread rather than a point.

In [ ]:
if not ok3:
    raise SystemExit('Finish TODO 3 first.')

seed_results, seed_preds = [], []
for s in [0, 1, 2]:
    # ---- TODO 4 ----------------------------------------------------------
    m_s = None        # build a model with seed=s
    h_s = None        # train it: 40 epochs, verbose=False, seed=s
    p_s = None        # predictions on the test set (use evaluate(), take [1], .numpy())
    # ----------------------------------------------------------------------
    if p_s is None:
        raise SystemExit('TODO 4 is not filled in yet.')
    p_s = np.asarray(p_s)
    a_s, f_s, r_s, _ = score(yte.numpy(), p_s)
    seed_results.append((a_s, f_s, r_s))
    seed_preds.append(p_s)
    print('seed %d   accuracy %.4f   macro-F1 %.4f' % (s, a_s, f_s))

accs = np.array([r[0] for r in seed_results])
f1s = np.array([r[1] for r in seed_results])

ok4 = all([
    check('three runs completed', len(seed_results) == 3),
    check('each run predicted every test row',
          all(len(p) == len(yte) for p in seed_preds),
          'p_s should be one predicted class per test row'),
    check('the three runs really are three different models',
          len({hash(p.tobytes()) for p in seed_preds}) > 1
          or len({float(a) for a in accs}) > 1,
          'pass seed=s to BOTH build_model and train_model, or all three runs '
          'are the same model trained twice over'),
    check('every run beat the majority baseline', accs.min() > majority_acc - 1e-9),
])

In [ ]:
if not ok4:
    raise SystemExit('Finish TODO 4 first.')

a_lin, f_lin, r_lin, _ = score(yte.numpy(), pred_linear)

print('=' * 66)
print('%-28s %10s %10s' % ('model', 'accuracy', 'macro-F1'))
print('-' * 66)
print('%-28s %10.4f %10.4f' % ('always "%s"' % CLASSES[majority_class][:14],
                               majority_acc,
                               f1_score(yte.numpy(),
                                        np.full_like(yte.numpy(), majority_class),
                                        average='macro')))
print('%-28s %10.4f %10.4f' % ('straight boundary', a_lin, f_lin))
print('%-28s %10.4f %10.4f' % ('dense net (mean of 3 seeds)', accs.mean(), f1s.mean()))
print('%-28s %10.4f %10.4f' % ('   spread across seeds', accs.max() - accs.min(),
                               f1s.max() - f1s.min()))
print('=' * 66)

gap = accs.mean() - a_lin
spread = accs.max() - accs.min()
print()
print('net minus straight boundary:  accuracy %+.4f   macro-F1 %+.4f'
      % (gap, f1s.mean() - f_lin))
print('seed-to-seed spread of the net: %.4f' % spread)
print()
if abs(gap) <= max(spread, 1e-4):
    print('VERDICT: the difference is NO BIGGER than your own run-to-run spread.')
    print('         You cannot claim a winner on accuracy from this evidence, and')
    print('         saying so plainly is the right answer.')
elif gap > 0:
    print('VERDICT: the network beats the straight-boundary model, and the margin')
    print('         is larger than the seed spread.')
else:
    print('VERDICT: the straight-boundary model BEATS the network here, by more than')
    print('         the seed spread. That is a real result. Report it.')
print()
print('Now look at macro-F1 as well as accuracy. On an imbalanced track they can')
print('disagree sharply, and macro-F1 is the one that reflects the rare classes.')

### Figure 3 — where the leak would have been *(written for you)*

Your bundle holds out **whole subjects**. The obvious alternative — shuffling rows and splitting at
random — lets windows from the same recording land on both sides, so the model can recognise the
subject instead of the condition.

How much does that inflate the score? On some tracks a lot, on others almost nothing — and on a
track where every subject contributes exactly one row there is nothing to leak in the first place,
because the two ways of splitting are the same split. Run it and find out which yours is.
**This figure is the one most likely to end up in your memo.**

In [ ]:
if not ok4:
    raise SystemExit('Finish TODO 4 first.')

from sklearn.model_selection import train_test_split

rtr, rte = train_test_split(np.arange(len(y)), test_size=len(test_idx) / len(y),
                            random_state=0, stratify=y)
sc2 = StandardScaler().fit(X[rtr])
Xtr2 = torch.tensor(sc2.transform(X[rtr]), dtype=torch.float32)
Xte2 = torch.tensor(sc2.transform(X[rte]), dtype=torch.float32)
ytr2 = torch.tensor(y[rtr], dtype=torch.long)
yte2 = torch.tensor(y[rte], dtype=torch.long)

m_leak = build_model(Xtr2.shape[1], K, seed=0)
train_model(m_leak, Xtr2, ytr2, Xte2, yte2, epochs=40, verbose=False)
_, p_leak = evaluate(m_leak, Xte2, yte2, nn.CrossEntropyLoss())
a_leak, f_leak, _, _ = score(yte2.numpy(), p_leak.numpy())

shared = len(np.intersect1d(subject[rtr], subject[rte]))
pct = 100 * shared / len(np.unique(subject))

print('subject-wise split (your bundle) : accuracy %.4f   macro-F1 %.4f'
      % (accs.mean(), f1s.mean()))
print('random row split                 : accuracy %.4f   macro-F1 %.4f'
      % (a_leak, f_leak))
print('inflation from the leak          : %+.4f accuracy' % (a_leak - accs.mean()))
print()
n_subj = len(np.unique(subject))
rows_per_subject = len(y) / n_subj
print('%d of %d subjects (%.0f%%) appear on BOTH sides of the random split.'
      % (shared, n_subj, pct))
if rows_per_subject < 1.5:
    print()
    print('NOTE: this track has %.1f rows per subject - every record is its own subject.'
          % rows_per_subject)
    print('      There is nothing for a random split to leak, and the two bars should')
    print('      agree to within the seed spread. That null result IS your answer to')
    print('      question 4, and it is worth saying why: subject-level splitting only')
    print('      buys you anything when one subject contributes many rows.')

fig, ax = plt.subplots(figsize=(5.6, 3.2))
bars = ax.bar(['subject-wise\n(honest)', 'random rows\n(leaky)'],
              [accs.mean(), a_leak], color=['#1F3864', '#9C2B2B'], width=.55)
ax.axhline(majority_acc, color='#2E6B4F', ls='--', lw=1.2)
ax.text(1.42, majority_acc, ' majority', color='#2E6B4F', fontsize=8, va='center')
ax.axhline(a_lin, color='grey', ls=':', lw=1.2)
ax.text(1.42, a_lin, ' straight\n boundary', color='grey', fontsize=8, va='center')
for b_, v in zip(bars, [accs.mean(), a_leak]):
    ax.text(b_.get_x() + b_.get_width() / 2, v + .008, '%.4f' % v, ha='center', fontsize=10)
ax.set_ylabel('test accuracy'); ax.set_ylim(0, 1.08)
ax.set_title('same model, same data, two ways of splitting')
plt.tight_layout(); plt.show()

---
## Step 5 · **TODO 5** — buying recall on the rare class

If your per-class recalls in Step 3 were uneven, you have a choice to make. `CrossEntropyLoss`
takes a `weight` argument — one number per class — that multiplies each example's loss by its
class's weight, so mistakes on rare classes cost more.

The standard choice is **inverse frequency**: `n_total / (K * n_c)`, computed on the **training**
labels only.

Train one more model with those weights and compare. What you should see is not a free win.

In [ ]:
if not ok4:
    raise SystemExit('Finish TODO 4 first.')

# ---- TODO 5 ----------------------------------------------------------
class_weights = None      # a torch.float32 tensor of length K, inverse frequency on ytr
model_w = None            # a fresh model (seed=0)
hist_w = None             # train it with weight=class_weights, 40 epochs, verbose=False
# ----------------------------------------------------------------------
if class_weights is None or model_w is None:
    raise SystemExit('TODO 5 is not filled in yet.')

_, pred_w = evaluate(model_w, Xte, yte, nn.CrossEntropyLoss())
a_w, f_w, r_w, cm_w = score(yte.numpy(), pred_w.numpy())

ref_w = len(ytr) / (K * np.bincount(ytr.numpy(), minlength=K).clip(1))
ok5 = all([
    check('weights are inverse frequency on the TRAINING labels',
          torch.is_tensor(class_weights) and len(class_weights) == K
          and np.allclose(class_weights.numpy(), ref_w, rtol=1e-4),
          'n_total / (K * count_c), counted on ytr - not on the whole dataset'),
    check('a weighted model was trained', hist_w is not None and len(hist_w) == 40),
])

if ok5:
    print()
    print('%-16s %14s %14s' % ('class', 'recall before', 'recall after'))
    for c in range(K):
        print('%-16s %14.3f %14.3f' % (CLASSES[c], rec[c], r_w[c]))
    print()
    print('%-16s %14.4f %14.4f' % ('accuracy', acc, a_w))
    print('%-16s %14.4f %14.4f' % ('macro-F1', mf1, f_w))
    print()
    print('class weights used:', class_weights.numpy().round(3))

Look at that table carefully before you write about it. On most tracks the rare-class recall goes
**up** and the overall accuracy goes **down** — you bought sensitivity with specificity. Whether
that is the right trade is a clinical question about the relative cost of a missed case against a
false alarm, and it is not one your loss function can answer.

---
## Step 6 · The memo

Write **`REPORT.md`**, one page, answering these six. Short answers, real numbers, no padding.

1. **What is your track, and what is the bar?** Name the dataset, the number of rows, features and
   classes, and the two baselines with their accuracy *and* macro-F1.
2. **Did the network beat the straight-boundary model?** Quote the mean over three seeds and the
   spread. If the margin is smaller than the spread, say so — that is a finding, not a failure.
3. **Where do the errors go?** Use the confusion matrix and the per-class recalls. Which class is
   the model worst at, and is that the class it would matter most to get right?
4. **How much would a careless split have flattered you?** Quote both numbers from Figure 3 and the
   fraction of subjects that appear on both sides. Explain in one sentence *why* the effect is the
   size it is on your data.
5. **What did class weighting cost and buy?** Give the before/after recall for the rarest class and
   the change in overall accuracy. State which you would ship and why.
6. **What would you do with another two weeks?** One concrete, specific thing — not "try a bigger
   network".

### What we are grading

| | |
|---|---|
| The five `TODO` cells run and every check passes | 30 |
| The comparison is done honestly: three seeds, spread reported, verdict stated | 25 |
| Per-class analysis, not just accuracy | 20 |
| The split experiment is run and correctly interpreted | 15 |
| Memo is one page, quantitative, and answers all six | 10 |

**You are not graded on getting a high accuracy.** You are graded on knowing what your number
means. A careful report that concludes "the network is not worth it on this data" scores full
marks; an unexamined 0.99 does not.

In [ ]:
# final check - everything above should be OK before you submit
steps = {'Step 1  build the network': ok1,
         'Step 2  training loop': ok2,
         'Step 3  proper scoring': ok3,
         'Step 4  comparison across seeds': ok4,
         'Step 5  class weighting': ok5}
for k, v in steps.items():
    print('  %s  %s' % ('OK  ' if v else 'NOT YET', k))
print()
print('All steps pass. Write REPORT.md and AI_USE.md.' if all(steps.values())
      else 'Finish the steps marked NOT YET.')

---
## If you get stuck

| Symptom | Almost always |
|---|---|
| `mat1 and mat2 shapes cannot be multiplied` | the first `Linear` width does not match `X.shape[1]` |
| `mat1 and mat2 must have the same dtype` | a `float64` tensor reached a layer — build tensors with `dtype=torch.float32` |
| `Expected input batch_size (a) to match target batch_size (b)` | logits and labels have different lengths |
| Loss does not move at all | the five lines of TODO 2 are missing, or `zero_grad()` is absent |
| Loss explodes after a few steps | `optimiser.zero_grad()` is missing — gradients are summing |
| Accuracy is high but macro-F1 is terrible | the model predicts the majority class nearly always; look at recall per class |
| Two runs of the same cell give different numbers | that is expected — it is exactly why Step 4 asks for three seeds |
| The random-row split scores *worse* than the subject split | possible on a track where each row is an independent patient; say so in the memo |